# Carga del almacén de Grano de Oro

Carga del almacén de Grano de Oro

Traducción a Python del job «job_carga_dw» (pyspark).
Origen: job_carga_dw.kjb
Generado por Pentaho Live. Entradas: 11 (10 exactas, 1 aproximadas, 0 a completar); transformaciones: 5 (43 steps, 0 a completar).
Ejecútalo como Kitchen: python script.py (código de salida 0 si el job termina bien).

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T, Window
import os
import logging
import sys
import contextlib
import functools
import pandas as pd
import json

_builder = SparkSession.builder.appName("job_carga_dw")
if os.name == "nt":  # Windows: Spark local en localhost (un nombre de equipo con _ lo rompe)
    _builder = _builder.config("spark.driver.host", "localhost").config("spark.driver.bindAddress", "127.0.0.1")
spark = _builder.getOrCreate()

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("job_carga_dw")

# carpeta del script (o del notebook): la misma que la del .kjb
AQUI = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
VARS = {
    "Internal.Job.Filename.Directory": os.environ.get("Internal.Job.Filename.Directory", AQUI),
    "Internal.Entry.Current.Directory": os.environ.get("Internal.Entry.Current.Directory", AQUI),
    "Internal.Job.Name": "job_carga_dw",
    "Internal.Transformation.Filename.Directory": os.environ.get("Internal.Transformation.Filename.Directory", AQUI),
}
# filas que una transformación deja en el resultado (Copy rows to result) para la siguiente
RESULT = {}

# PySpark necesita el .jar del driver JDBC: spark.jars o --jars al lanzar
jdbc_dw_grano_de_oro = {"url": f"jdbc:sqlite:{VARS['Internal.Entry.Current.Directory']}/../salida/grano_de_oro_dw.sqlite", "driver": "org.sqlite.JDBC"}


class Abortar(Exception):
    """Abort job: corta el job entero (main() o el sub-job devuelve False)."""


def entrada(f):
    """Una entrada del job: devuelve True (éxito) o False (fallo); una excepción cuenta como fallo, como en PDI."""
    @functools.wraps(f)
    def envoltura():
        log.info("%s", f.__doc__.splitlines()[0] if f.__doc__ else f.__name__)
        try:
            return f() is not False
        except Abortar:
            raise
        except Exception:
            log.exception("Falló %s", f.__name__)
            return False
    return envoltura


@contextlib.contextmanager
def con_variables(valores):
    """Da valor a variables (los parámetros de una entrada) mientras dura el bloque."""
    antes = {k: VARS[k] for k in valores if k in VARS}
    VARS.update(valores)
    try:
        yield
    finally:
        for k in valores:
            if k in antes:
                VARS[k] = antes[k]
            else:
                VARS.pop(k, None)


def sentencias_sql(sql):
    """Separa un script SQL en sentencias por «;» (fuera de comillas), como Execute SQL script."""
    partes, actual, comilla = [], [], None
    for ch in sql:
        if comilla:
            comilla = None if ch == comilla else comilla
        elif ch in "'\"":
            comilla = ch
        elif ch == ";":
            partes.append("".join(actual))
            actual = []
            continue
        actual.append(ch)
    partes.append("".join(actual))
    return [p.strip() for p in partes if p.strip()]


def ejecutar_sql(jdbc, sql, una_sola=False):
    """Ejecuta un script SQL por JDBC con la JVM de Spark (el driver JDBC tiene que estar en su classpath)."""
    con = spark._jvm.java.sql.DriverManager.getConnection(jdbc["url"], jdbc["user"], jdbc["password"])
    try:
        st = con.createStatement()
        for sentencia in [sql] if una_sola else sentencias_sql(sql):
            st.execute(sentencia)
    finally:
        con.close()


def _jsonpath(doc, ruta):
    """Valores de un JSONPath sencillo ($.a.b, $..a, [*], [n], ['x']) en orden de documento."""
    import re
    pasos = re.findall(r"\.\.[^.\[]+|\.[^.\[]+|\[[^\]]*\]", ruta.lstrip("$"))
    nodos = [doc]
    for p in pasos:
        sig = []
        if p.startswith(".."):
            clave = p[2:]
            def bajar(n):
                if isinstance(n, dict):
                    if clave in n:
                        sig.append(n[clave])
                    for v in n.values():
                        bajar(v)
                elif isinstance(n, list):
                    for v in n:
                        bajar(v)
            for n in nodos:
                bajar(n)
        elif p in (".*", "[*]"):
            for n in nodos:
                sig.extend(n if isinstance(n, list) else n.values() if isinstance(n, dict) else [])
        elif p.startswith("["):
            k = p[1:-1].strip("'\"")
            for n in nodos:
                if isinstance(n, list) and k.lstrip("-").isdigit() and -len(n) <= int(k) < len(n):
                    sig.append(n[int(k)])
                elif isinstance(n, dict) and k in n:
                    sig.append(n[k])
        else:
            for n in nodos:
                if isinstance(n, dict) and p[1:] in n:
                    sig.append(n[p[1:]])
        nodos = sig
    return nodos


def pdi_rachas(df, claves):
    """Añade _orden (orden de llegada) y _racha: el número de racha de filas seguidas con las mismas claves, como agrupa PDI (compara cada fila con la anterior; nulo = nulo)."""
    df = df.withColumn("_orden", F.monotonically_increasing_id())
    w = Window.orderBy("_orden")
    igual = F.lag("_orden").over(w).isNotNull()
    for c in claves:
        igual = igual & F.col(c).eqNullSafe(F.lag(c).over(w))
    return df.withColumn("_racha", F.sum(F.when(igual, 0).otherwise(1)).over(w))


def _numero_texto(serie, decimales, fijos, agrupar, sep_decimal=".", sep_grupo=","):
    """Número a texto como una máscara de Java (#,##0.00): decimales, mínimos fijos y separadores."""
    def f(v):
        if pd.isna(v):
            return None
        t = format(float(v), ("," if agrupar else "") + f".{decimales}f")
        if fijos < decimales and "." in t:
            ent, dec = t.split(".")
            dec = dec[:fijos] + dec[fijos:].rstrip("0")
            t = ent + ("." + dec if dec else "")
        return t.replace(",", "\0").replace(".", sep_decimal).replace("\0", sep_grupo)
    return serie.map(f)


def _escribir_texto(df, ruta, sep=";", encierro='"', forzar=False, cabecera=True, anexar=False,
                    codificacion="utf-8", fin="\r\n", textos=()):
    """Escribe df como el Text file output de PDI: encierra los campos de texto (siempre si forzar;
    si no, sólo los que llevan el separador, el encierro o un salto de línea) y pone la cabecera si el
    archivo es nuevo."""
    nuevo = not (anexar and os.path.exists(ruta) and os.path.getsize(ruta) > 0)
    columnas = []
    for c in df.columns:
        s = df[c].astype(object).map(lambda v: "" if v is None or pd.isna(v) else str(v))
        if c in textos and encierro:
            if forzar:
                poner = s != ""
            else:
                poner = s.str.contains(sep, regex=False) | s.str.contains(encierro, regex=False) \
                    | s.str.contains("\n", regex=False)
            s = s.where(~poner, encierro + s.str.replace(encierro, encierro * 2, regex=False) + encierro)
        columnas.append(s)
    with open(ruta, "a" if anexar else "w", encoding=codificacion, newline="") as f:
        if cabecera and nuevo:
            f.write(sep.join(encierro + str(c) + encierro if forzar and c in textos else str(c)
                             for c in df.columns) + fin)
        if columnas:
            lineas = columnas[0]
            for s in columnas[1:]:
                lineas = lineas + sep + s
            f.write("".join(linea + fin for linea in lineas))


def _nombre_salida(nombre, extension="", fecha="", parte=None):
    """Nombre del archivo como lo arma PDI: nombre[fecha][_parte].extensión."""
    import datetime
    r = nombre + (datetime.datetime.now().strftime(fecha) if fecha else "")
    if parte is not None:
        r += f"_{parte}"
    return r + ("." + extension if extension else "")

## Comprobar ficheros de entrada

`FILES_EXIST`

Checks if files exist: éxito si existen todos los archivos. Los tres ficheros de datos/ tienen que existir.

In [ ]:
@entrada
def e_comprobar_ficheros_de_entrada():
    """Comprobar ficheros de entrada (FILES_EXIST): Checks if files exist: éxito si existen todos los archivos. Los tres ficheros de datos/ tienen que existir."""
    rutas = [
        f"{VARS['Internal.Entry.Current.Directory']}/../datos/ventas_2026_09.csv",
        f"{VARS['Internal.Entry.Current.Directory']}/../datos/productos.xlsx",
        f"{VARS['Internal.Entry.Current.Directory']}/../datos/tiendas.json",
    ]
    faltan = [r for r in rutas if not os.path.exists(r)]
    for r in faltan:
        log.warning("No existe %s", r)
    return not faltan

## Crear carpeta salida

`CREATE_FOLDER`

Create a folder: crea la carpeta. Carpeta del almacén SQLite, los rechazos y los KPI.

In [ ]:
@entrada
def e_crear_carpeta_salida():
    """Crear carpeta salida (CREATE_FOLDER): Create a folder: crea la carpeta. Carpeta del almacén SQLite, los rechazos y los KPI."""
    carpeta = f"{VARS['Internal.Entry.Current.Directory']}/../salida"
    if os.path.isdir(carpeta):
        return True
    os.makedirs(carpeta)
    return True

## Crear modelo estrella

`SQL` · **aproximado**

Execute SQL script: ejecuta sentencias SQL en la base. Borra y vuelve a crear las tablas: la carga es completa e idempotente.

> Nota: PySpark no ejecuta SQL de modificación: se usa JDBC con la JVM de Spark (spark._jvm); el driver tiene que estar en spark.driver.extraClassPath.

In [ ]:
@entrada
def e_crear_modelo_estrella():
    """Crear modelo estrella (SQL): Execute SQL script: ejecuta sentencias SQL en la base. Borra y vuelve a crear las tablas: la carga es completa e idempotente."""
    # estado: aproximado
    # NOTA: PySpark no ejecuta SQL de modificación: se usa JDBC con la JVM de Spark (spark._jvm); el driver tiene que estar en spark.driver.extraClassPath.
    sql = """
    DROP TABLE IF EXISTS hechos_ventas;
    DROP TABLE IF EXISTS kpi_ventas_tienda_categoria;
    DROP TABLE IF EXISTS kpi_ticket_medio_tienda;
    DROP TABLE IF EXISTS dim_tienda;
    DROP TABLE IF EXISTS dim_producto;
    DROP TABLE IF EXISTS dim_fecha;

    CREATE TABLE dim_tienda (
      sk_tienda      INTEGER PRIMARY KEY,
      id_tienda      TEXT NOT NULL UNIQUE,
      nombre         TEXT NOT NULL,
      barrio         TEXT,
      distrito       TEXT,
      codigo_postal  TEXT,
      fecha_apertura TEXT,
      superficie_m2  INTEGER,
      plazas         INTEGER
    );

    CREATE TABLE dim_producto (
      sk_producto     INTEGER PRIMARY KEY,
      codigo_producto TEXT NOT NULL UNIQUE,
      nombre          TEXT NOT NULL,
      categoria       TEXT NOT NULL,
      precio_venta    REAL
    );

    CREATE TABLE dim_fecha (
      sk_fecha         INTEGER PRIMARY KEY,
      fecha            TEXT NOT NULL UNIQUE,
      anio             INTEGER NOT NULL,
      mes              INTEGER NOT NULL,
      dia              INTEGER NOT NULL,
      dia_semana       INTEGER NOT NULL,
      nombre_dia       TEXT NOT NULL,
      es_fin_de_semana TEXT NOT NULL
    );

    CREATE TABLE hechos_ventas (
      id_ticket       TEXT    NOT NULL,
      linea           INTEGER NOT NULL,
      sk_fecha        INTEGER NOT NULL REFERENCES dim_fecha (sk_fecha),
      sk_tienda       INTEGER NOT NULL REFERENCES dim_tienda (sk_tienda),
      sk_producto     INTEGER NOT NULL REFERENCES dim_producto (sk_producto),
      cantidad        INTEGER NOT NULL,
      precio_unitario REAL    NOT NULL,
      importe         REAL    NOT NULL,
      metodo_pago     TEXT    NOT NULL,
      PRIMARY KEY (id_ticket, linea)
    );

    CREATE TABLE kpi_ventas_tienda_categoria (
      tienda    TEXT NOT NULL,
      categoria TEXT NOT NULL,
      unidades  INTEGER NOT NULL,
      importe   REAL NOT NULL,
      PRIMARY KEY (tienda, categoria)
    );

    CREATE TABLE kpi_ticket_medio_tienda (
      tienda        TEXT PRIMARY KEY,
      importe_total REAL NOT NULL,
      num_tickets   INTEGER NOT NULL,
      ticket_medio  REAL NOT NULL
    );
    """
    ejecutar_sql(jdbc_dw_grano_de_oro, sql)
    return True

## Cargar dim_tienda

`TRANS`

Transformación «01_dim_tienda» (01_dim_tienda.ktr): 5 steps.

In [ ]:
@entrada
def tr_cargar_dim_tienda():
    """Cargar dim_tienda (TRANS): Transformación «01_dim_tienda» (01_dim_tienda.ktr): 5 steps."""
    # --- 1. Localizar tiendas.json (GetFileNames)
    # Step de tipo GetFileNames
    # NOTA: lastmodifiedtime va en UTC; la fecha de PDI es la local.
    archivos_localizar_tiendas_json = [f"{VARS['Internal.Entry.Current.Directory']}/../datos/tiendas.json"]
    pdf_localizar_tiendas_json = pd.DataFrame([{
        "filename": os.path.abspath(a), "short_filename": os.path.basename(a),
        "path": os.path.dirname(os.path.abspath(a)), "type": "file" if os.path.isfile(a) else "folder",
        "exists": os.path.exists(a), "ishidden": os.path.basename(a).startswith("."),
        "isreadable": os.access(a, os.R_OK), "iswriteable": os.access(a, os.W_OK),
        "lastmodifiedtime": pd.Timestamp(os.path.getmtime(a), unit="s"),
        "size": os.path.getsize(a), "extension": os.path.splitext(a)[1].lstrip("."),
        "uri": "file:///" + os.path.abspath(a).replace(os.sep, "/").lstrip("/"),
        "rooturi": "file:///",
    } for a in archivos_localizar_tiendas_json if os.path.isfile(a)])
    df_localizar_tiendas_json = spark.createDataFrame(pdf_localizar_tiendas_json)

    # --- 2. Leer tiendas (JSON) (JsonInput)
    # Step de tipo JsonInput
    # NOTA: El JSON sale del campo "filename" de la entrada (los demás campos no se conservan).
    # NOTA: El JSON se lee con Python y se pasa a Spark (spark.read.json necesita un objeto por línea).
    # NOTA: _jsonpath cubre los JSONPath habituales; los filtros [?(...)] hay que hacerlos a mano.
    def _filas_json(doc):
        cols = {"id_tienda": _jsonpath(doc, "$.tiendas[*].id_tienda"), "nombre": _jsonpath(doc, "$.tiendas[*].nombre"), "barrio": _jsonpath(doc, "$.tiendas[*].ubicacion.barrio"), "distrito": _jsonpath(doc, "$.tiendas[*].ubicacion.distrito"), "codigo_postal": _jsonpath(doc, "$.tiendas[*].ubicacion.codigo_postal"), "fecha_apertura": _jsonpath(doc, "$.tiendas[*].fecha_apertura"), "superficie_m2": _jsonpath(doc, "$.tiendas[*].superficie_m2"), "plazas": _jsonpath(doc, "$.tiendas[*].plazas")}
        n = max((len(v) for v in cols.values()), default=0)
        return pd.DataFrame({k: (v + [None] * n)[:n] for k, v in cols.items()}, dtype=object)
    pdf_leer_tiendas_json = pd.concat([_filas_json(json.load(open(t, encoding="utf-8"))) for t in [r["filename"] for r in df_localizar_tiendas_json.select("filename").collect()]], ignore_index=True)
    pdf_leer_tiendas_json = pdf_leer_tiendas_json.astype(object).where(pdf_leer_tiendas_json.notna(), None)
    pdf_leer_tiendas_json[["id_tienda", "nombre", "barrio", "distrito", "codigo_postal", "fecha_apertura", "superficie_m2", "plazas"]] = pdf_leer_tiendas_json[["id_tienda", "nombre", "barrio", "distrito", "codigo_postal", "fecha_apertura", "superficie_m2", "plazas"]].map(lambda v: v if v is None or isinstance(v, str) else json.dumps(v) if isinstance(v, (dict, list)) else str(v).lower() if isinstance(v, bool) else str(v))
    df_leer_tiendas_json = spark.createDataFrame(pdf_leer_tiendas_json, "`id_tienda` STRING, `nombre` STRING, `barrio` STRING, `distrito` STRING, `codigo_postal` STRING, `fecha_apertura` STRING, `superficie_m2` STRING, `plazas` STRING")
    df_leer_tiendas_json = df_leer_tiendas_json.withColumns({
        "superficie_m2": F.col("superficie_m2").cast("long"),
        "plazas": F.col("plazas").cast("long"),
    })

    # --- 3. Estandarizar textos (StringOperations)
    # Limpia campos de texto
    df_estandarizar_textos = (df_leer_tiendas_json
        .withColumn("id_tienda", F.upper(F.trim(F.col("id_tienda"))))
        .withColumn("nombre", F.trim(F.col("nombre")))
        .withColumn("barrio", F.trim(F.col("barrio")))
        .withColumn("distrito", F.trim(F.col("distrito"))))

    # --- 4. Asignar clave subrogada (Sequence)
    # Step de tipo Sequence
    df_asignar_clave_subrogada = df_estandarizar_textos.withColumn("sk_tienda", 1 + (F.row_number().over(Window.orderBy(F.monotonically_increasing_id())) - 1) * 1)

    # --- 5. Cargar dim_tienda (TableOutput)
    # Escribe en la tabla dim_tienda (dw_grano_de_oro)
    # NOTA: Spark necesita el .jar del driver JDBC (spark.jars).
    df_cargar_dim_tienda = df_asignar_clave_subrogada
    (df_asignar_clave_subrogada.select("sk_tienda", "id_tienda", "nombre", "barrio", "distrito", "codigo_postal", "fecha_apertura", "superficie_m2", "plazas").write.format("jdbc").options(**jdbc_dw_grano_de_oro)
        .option("dbtable", "dim_tienda")
        .option("batchsize", 1000).option("truncate", True).mode("overwrite").save())

## Cargar dim_producto

`TRANS`

Transformación «02_dim_producto» (02_dim_producto.ktr): 5 steps.

In [ ]:
@entrada
def tr_cargar_dim_producto():
    """Cargar dim_producto (TRANS): Transformación «02_dim_producto» (02_dim_producto.ktr): 5 steps."""
    # --- 1. Leer productos (Excel) (ExcelInput)
    # Lee productos.xlsx (4 campos)
    # NOTA: Spark no lee Excel: se lee con pandas (openpyxl) y se pasa a Spark.
    # NOTA: Necesita openpyxl (xlsx) o xlrd (xls antiguos).
    archivos_leer_productos_excel = [f"{VARS['Internal.Entry.Current.Directory']}/../datos/productos.xlsx"]
    partes = []
    for archivo in archivos_leer_productos_excel:
        for hoja, fila0, col0 in [("Productos", 0, 0)]:
            parte = pd.read_excel(archivo, sheet_name=hoja, header=None, skiprows=fila0 + 1, dtype=object)
            parte = parte.iloc[:, col0:col0 + 4]
            parte.columns = ["codigo_producto", "nombre", "categoria", "precio_venta"][:parte.shape[1]]
            partes.append(parte)
    pdf_leer_productos_excel = pd.concat(partes, ignore_index=True)
    pdf_leer_productos_excel = pdf_leer_productos_excel.dropna(how="all", subset=["codigo_producto", "nombre", "categoria", "precio_venta"]).reset_index(drop=True)
    pdf_leer_productos_excel["codigo_producto"] = pdf_leer_productos_excel["codigo_producto"].astype("string")
    pdf_leer_productos_excel["nombre"] = pdf_leer_productos_excel["nombre"].astype("string")
    pdf_leer_productos_excel["categoria"] = pdf_leer_productos_excel["categoria"].astype("string")
    pdf_leer_productos_excel["precio_venta"] = pd.to_numeric(pdf_leer_productos_excel["precio_venta"]).astype("float64")
    df_leer_productos_excel = spark.createDataFrame(pdf_leer_productos_excel.astype(object).where(pdf_leer_productos_excel.notna(), None))

    # --- 2. Estandarizar textos (StringOperations)
    # Limpia campos de texto
    df_estandarizar_textos = (df_leer_productos_excel
        .withColumn("codigo_producto", F.upper(F.trim(F.col("codigo_producto"))))
        .withColumn("nombre", F.trim(F.col("nombre")))
        .withColumn("categoria", F.initcap(F.trim(F.col("categoria")))))

    # --- 3. Ordenar por código (SortRows)
    # Ordena por codigo_producto
    # NOTA: Spark no garantiza un orden estable entre filas con la misma clave.
    df_ordenar_por_codigo = df_estandarizar_textos.orderBy(F.lower(F.col("codigo_producto")).asc_nulls_first())

    # --- 4. Asignar clave subrogada (Sequence)
    # Step de tipo Sequence
    df_asignar_clave_subrogada = df_ordenar_por_codigo.withColumn("sk_producto", 1 + (F.row_number().over(Window.orderBy(F.monotonically_increasing_id())) - 1) * 1)

    # --- 5. Cargar dim_producto (TableOutput)
    # Escribe en la tabla dim_producto (dw_grano_de_oro)
    # NOTA: Spark necesita el .jar del driver JDBC (spark.jars).
    df_cargar_dim_producto = df_asignar_clave_subrogada
    (df_asignar_clave_subrogada.select("sk_producto", "codigo_producto", "nombre", "categoria", "precio_venta").write.format("jdbc").options(**jdbc_dw_grano_de_oro)
        .option("dbtable", "dim_producto")
        .option("batchsize", 1000).option("truncate", True).mode("overwrite").save())

## Cargar dim_fecha

`TRANS`

Transformación «03_dim_fecha» (03_dim_fecha.ktr): 6 steps.

In [ ]:
@entrada
def tr_cargar_dim_fecha():
    """Cargar dim_fecha (TRANS): Transformación «03_dim_fecha» (03_dim_fecha.ktr): 6 steps."""
    # --- 1. Generar 30 días (RowGenerator)
    # Genera 30 filas
    df_generar_30_dias = spark.range(30).select(
    F.lit("2026-09-01").cast("string").alias("fecha_inicio")
)
    df_generar_30_dias = df_generar_30_dias.withColumns({
        "fecha_inicio": F.to_timestamp(F.col("fecha_inicio"), "yyyy-MM-dd"),
    })

    # --- 2. Numerar días (Sequence)
    # Step de tipo Sequence
    df_numerar_dias = df_generar_30_dias.withColumn("desplazamiento", 0 + (F.row_number().over(Window.orderBy(F.monotonically_increasing_id())) - 1) * 1)

    # --- 3. Calcular fecha (Calculator)
    # Calcula fecha, anio, mes, dia
    df_calcular_fecha = (df_numerar_dias
        .withColumn("fecha", (F.col("fecha_inicio") + F.make_dt_interval(days=F.col("desplazamiento"))))
        .withColumn("anio", F.year(F.col("fecha")))
        .withColumn("mes", F.month(F.col("fecha")))
        .withColumn("dia", F.dayofmonth(F.col("fecha"))))

    # --- 4. Atributos del día (Formula)
    # Calcula sk_fecha, dia_semana, nombre_dia, es_fin_de_semana
    df_atributos_del_dia = (df_calcular_fecha
        .withColumn("sk_fecha", (((F.col("anio") * 10000) + (F.col("mes") * 100)) + F.col("dia")))
        .withColumn("dia_semana", ((F.dayofweek(F.col("fecha")) + 5) % 7 + 1))
        .withColumn("nombre_dia", F.element_at(F.array(F.lit("lunes"), F.lit("martes"), F.lit("miércoles"), F.lit("jueves"), F.lit("viernes"), F.lit("sábado"), F.lit("domingo")), ((F.dayofweek(F.col("fecha")) + 5) % 7 + 1).cast("int")))
        .withColumn("es_fin_de_semana", F.when((((F.dayofweek(F.col("fecha")) + 5) % 7 + 1) > 5), "S").otherwise("N")))

    # --- 5. Fecha como texto (SelectValues)
    # Quita 2 campos
    df_fecha_como_texto = df_atributos_del_dia.drop("fecha_inicio", "desplazamiento")
    df_fecha_como_texto = df_fecha_como_texto.withColumn("fecha", F.date_format(F.col("fecha"), "yyyy-MM-dd"))

    # --- 6. Cargar dim_fecha (TableOutput)
    # Escribe en la tabla dim_fecha (dw_grano_de_oro)
    # NOTA: Spark necesita el .jar del driver JDBC (spark.jars).
    df_cargar_dim_fecha = df_fecha_como_texto
    (df_fecha_como_texto.select("sk_fecha", "fecha", "anio", "mes", "dia", "dia_semana", "nombre_dia", "es_fin_de_semana").write.format("jdbc").options(**jdbc_dw_grano_de_oro)
        .option("dbtable", "dim_fecha")
        .option("batchsize", 1000).option("truncate", True).mode("overwrite").save())

## Cargar hechos_ventas

`TRANS`

Transformación «04_hechos_ventas» (04_hechos_ventas.ktr): 16 steps.

In [ ]:
@entrada
def tr_cargar_hechos_ventas():
    """Cargar hechos_ventas (TRANS): Transformación «04_hechos_ventas» (04_hechos_ventas.ktr): 16 steps."""
    # --- 1. Leer ventas (CSV) (CsvInput)
    # Lee ventas_2026_09.csv (8 campos)
    # NOTA: Como en PDI, el texto vacío se lee como nulo.
    archivo_leer_ventas_csv = f"{VARS['Internal.Entry.Current.Directory']}/../datos/ventas_2026_09.csv"
    df_leer_ventas_csv = spark.read.csv(archivo_leer_ventas_csv, sep=";", quote="\"", header=True, encoding="UTF-8",
    schema="`id_ticket` STRING, `linea` STRING, `fecha` STRING, `id_tienda` STRING, `codigo_producto` STRING, `cantidad` STRING, `precio_unitario` STRING, `metodo_pago` STRING")
    df_leer_ventas_csv = df_leer_ventas_csv.withColumns({
        "linea": F.col("linea").cast("long"),
        "cantidad": F.col("cantidad").cast("long"),
    })

    # --- 2. Estandarizar textos (StringOperations)
    # Limpia campos de texto
    df_estandarizar_textos = (df_leer_ventas_csv
        .withColumn("id_tienda", F.upper(F.trim(F.col("id_tienda"))))
        .withColumn("codigo_producto", F.upper(F.trim(F.col("codigo_producto"))))
        .withColumn("metodo_pago", F.upper(F.trim(F.col("metodo_pago")))))

    # --- 3. Normalizar fecha y precio (ReplaceString)
    # Limpia campos de texto
    df_normalizar_fecha_y_precio = (df_estandarizar_textos
        .withColumn("fecha", F.regexp_replace(F.col("fecha"), "(?i)^(\\d{2})/(\\d{2})/(\\d{4})$", "$3-$2-$1"))
        .withColumn("precio_unitario", F.regexp_replace(F.col("precio_unitario"), "(?i),", ".")))

    # --- 4. Convertir precio a número (SelectValues)
    # Cambia el tipo de 1 campos
    df_convertir_precio_a_numero = df_normalizar_fecha_y_precio
    df_convertir_precio_a_numero = df_convertir_precio_a_numero.withColumn("precio_unitario", F.col("precio_unitario").cast("double"))

    # --- 5. Ordenar por ticket y línea (SortRows)
    # Ordena por id_ticket, linea
    # NOTA: Spark no garantiza un orden estable entre filas con la misma clave.
    df_ordenar_por_ticket_y_linea = df_convertir_precio_a_numero.orderBy(F.lower(F.col("id_ticket")).asc_nulls_first(), F.col("linea").asc_nulls_first())

    # --- 6. Eliminar duplicados (Unique)
    # Quita duplicados por id_ticket, linea
    # PDI compara cada fila con la anterior: sólo quita duplicados SEGUIDOS (entrada ordenada).
    d = pdi_rachas(df_ordenar_por_ticket_y_linea, ["id_ticket", "linea"])
    d = d.withColumn("_primera", F.col("_orden") == F.min("_orden").over(Window.partitionBy("_racha")))
    df_eliminar_duplicados = d.filter("_primera").orderBy("_orden").drop("_orden", "_racha", "_primera")
    df_eliminar_duplicados_error = d.filter(~F.col("_primera")).orderBy("_orden").drop("_orden", "_racha", "_primera")

    # --- 7. Buscar tienda (DBLookup)
    # Busca valores en la tabla dim_tienda
    # NOTA: En Spark la «primera» fila de la tabla según ORDER BY no está garantizada tras leer.
    # NOTA: La tabla se lee una vez entera (como la caché «cargar todo» de PDI).
    tabla = spark.read.format("jdbc").options(**jdbc_dw_grano_de_oro).option("query", "SELECT id_tienda, sk_tienda FROM dim_tienda").load()
    busca = tabla.withColumn("_orden", F.monotonically_increasing_id())
    busca = busca.withColumn("_ultima", F.min("_orden").over(Window.partitionBy("id_tienda")))
    busca = busca.filter(F.col("_orden") == F.col("_ultima")).select(F.col("id_tienda").alias("_k0"), F.col("sk_tienda").alias("sk_tienda"), F.lit(True).alias("_hay"))
    df_buscar_tienda = df_eliminar_duplicados.join(F.broadcast(busca), F.col("id_tienda").eqNullSafe(F.col("_k0")), "left")
    df_buscar_tienda = df_buscar_tienda.drop("_k0", "_hay")

    # --- 8. Marcar duplicado (Constant)
    # Agrega las constantes motivo_rechazo
    df_marcar_duplicado = (df_eliminar_duplicados_error
        .withColumn("motivo_rechazo", F.lit("DUPLICADO")))

    # --- 9. Buscar producto (DBLookup)
    # Busca valores en la tabla dim_producto
    # NOTA: En Spark la «primera» fila de la tabla según ORDER BY no está garantizada tras leer.
    # NOTA: La tabla se lee una vez entera (como la caché «cargar todo» de PDI).
    tabla_2 = spark.read.format("jdbc").options(**jdbc_dw_grano_de_oro).option("query", "SELECT codigo_producto, sk_producto FROM dim_producto").load()
    busca_2 = tabla_2.withColumn("_orden", F.monotonically_increasing_id())
    busca_2 = busca_2.withColumn("_ultima", F.min("_orden").over(Window.partitionBy("codigo_producto")))
    busca_2 = busca_2.filter(F.col("_orden") == F.col("_ultima")).select(F.col("codigo_producto").alias("_k0"), F.col("sk_producto").alias("sk_producto"), F.lit(True).alias("_hay"))
    df_buscar_producto = df_buscar_tienda.join(F.broadcast(busca_2), F.col("codigo_producto").eqNullSafe(F.col("_k0")), "left")
    df_buscar_producto = df_buscar_producto.drop("_k0", "_hay")

    # --- 10. Buscar fecha (DBLookup)
    # Busca valores en la tabla dim_fecha
    # NOTA: En Spark la «primera» fila de la tabla según ORDER BY no está garantizada tras leer.
    # NOTA: La tabla se lee una vez entera (como la caché «cargar todo» de PDI).
    tabla_3 = spark.read.format("jdbc").options(**jdbc_dw_grano_de_oro).option("query", "SELECT fecha, sk_fecha FROM dim_fecha").load()
    busca_3 = tabla_3.withColumn("_orden", F.monotonically_increasing_id())
    busca_3 = busca_3.withColumn("_ultima", F.min("_orden").over(Window.partitionBy("fecha")))
    busca_3 = busca_3.filter(F.col("_orden") == F.col("_ultima")).select(F.col("fecha").alias("_k0"), F.col("sk_fecha").alias("sk_fecha"), F.lit(True).alias("_hay"))
    df_buscar_fecha = df_buscar_producto.join(F.broadcast(busca_3), F.col("fecha").eqNullSafe(F.col("_k0")), "left")
    df_buscar_fecha = df_buscar_fecha.drop("_k0", "_hay")

    # --- 11. Clasificar línea (Janino)
    # Step de tipo Janino
    # NOTA: Con nulos, el código Java de PDI suele fallar; aquí da nulo.
    df_clasificar_linea = (df_buscar_fecha
        .withColumn("motivo_rechazo", F.when((F.col("id_tienda").isNull() | (F.length(F.col("id_tienda")) == 0)), "TIENDA_VACIA").otherwise(F.when((F.col("cantidad").isNull() | (F.col("cantidad").cast("long") <= 0)), "CANTIDAD_NO_VALIDA").otherwise(F.when(F.col("sk_tienda").isNull(), "TIENDA_DESCONOCIDA").otherwise(F.when(F.col("sk_producto").isNull(), "PRODUCTO_DESCONOCIDO").otherwise(F.when(F.col("sk_fecha").isNull(), "FECHA_NO_VALIDA").otherwise(None)))))))

    # --- 12. ¿Línea válida? (FilterRows)
    # Filtra filas con motivo_rechazo IS NULL
    cond_linea_valida = (F.col("motivo_rechazo").isNull() | (F.col("motivo_rechazo") == ""))
    df_linea_valida = df_clasificar_linea.filter(cond_linea_valida)
    df_linea_valida_false = df_clasificar_linea.filter(~cond_linea_valida)

    # --- 13. Preparar rechazo (SelectValues)
    # Se queda con 9 campos
    df_preparar_rechazo = df_linea_valida_false.select("id_ticket", "linea", "fecha", "id_tienda", "codigo_producto", "cantidad", "precio_unitario", "metodo_pago", "motivo_rechazo")

    # --- 14. Escribir rechazos (TextFileOutput)
    # Escribe rechazos_ventas.csv
    # NOTA: Spark escribe una carpeta con varios archivos; aquí se escribe un solo archivo como en PDI, pasando por pandas (junta las filas en el driver).
    df_escribir_rechazos_in = df_marcar_duplicado.unionByName(df_preparar_rechazo, allowMissingColumns=True)
    df_escribir_rechazos = df_escribir_rechazos_in
    pdf_escribir_rechazos = df_escribir_rechazos_in.toPandas()  # junta las filas en el driver
    sal_escribir_rechazos = pdf_escribir_rechazos[["id_ticket", "linea", "fecha", "id_tienda", "codigo_producto", "cantidad", "precio_unitario", "metodo_pago", "motivo_rechazo"]].copy()
    sal_escribir_rechazos["precio_unitario"] = _numero_texto(sal_escribir_rechazos["precio_unitario"], 2, 2, False, ".", ",")
    ruta_escribir_rechazos = _nombre_salida(f"{VARS['Internal.Entry.Current.Directory']}/../salida/rechazos_ventas", "csv")
    _escribir_texto(sal_escribir_rechazos, ruta_escribir_rechazos, sep=";", encierro="\"", fin="\n",
                    codificacion="UTF-8", textos=["id_ticket", "fecha", "id_tienda", "codigo_producto", "metodo_pago", "motivo_rechazo"])
    # en Spark distribuido (una carpeta con varios part-*.csv): df_escribir_rechazos.write.mode("overwrite").option("header", True).option("sep", ";").csv(...)

    # --- 15. Calcular importe (Calculator)
    # Calcula dos_decimales, importe_sin_redondear, importe
    df_calcular_importe = (df_linea_valida
        .withColumn("dos_decimales", F.lit(2))
        .withColumn("importe_sin_redondear", (F.col("precio_unitario") * F.col("cantidad")))
        .withColumn("importe", (F.signum(F.col("importe_sin_redondear")) * F.floor(F.abs(F.col("importe_sin_redondear")).cast("decimal(38,12)") * F.pow(F.lit(10), F.col("dos_decimales")).cast("decimal(38,12)") + 0.5) / F.pow(F.lit(10), F.col("dos_decimales")).cast("decimal(38,12)")).cast("double")))
    df_calcular_importe = df_calcular_importe.drop("dos_decimales", "importe_sin_redondear")  # campos temporales

    # --- 16. Cargar hechos_ventas (TableOutput)
    # Escribe en la tabla hechos_ventas (dw_grano_de_oro)
    # NOTA: Spark necesita el .jar del driver JDBC (spark.jars).
    df_cargar_hechos_ventas = df_calcular_importe
    (df_calcular_importe.select("id_ticket", "linea", "sk_fecha", "sk_tienda", "sk_producto", "cantidad", "precio_unitario", "importe", "metodo_pago").write.format("jdbc").options(**jdbc_dw_grano_de_oro)
        .option("dbtable", "hechos_ventas")
        .option("batchsize", 1000).option("truncate", True).mode("overwrite").save())

## Calcular KPI

`TRANS`

Transformación «05_kpi_ventas» (05_kpi_ventas.ktr): 11 steps.

In [ ]:
@entrada
def tr_calcular_kpi():
    """Calcular KPI (TRANS): Transformación «05_kpi_ventas» (05_kpi_ventas.ktr): 11 steps."""
    # --- 1. Leer modelo estrella (TableInput)
    # Lee la tabla hechos_ventas (dw_grano_de_oro)
    # NOTA: Los tipos los decide el driver de Python/JDBC: pueden diferir un poco de los de PDI.
    df_leer_modelo_estrella = (spark.read.format("jdbc").options(**jdbc_dw_grano_de_oro)
        .option("query", """SELECT t.nombre AS tienda,
       p.categoria,
       h.id_ticket,
       h.cantidad,
       h.importe
FROM hechos_ventas h
JOIN dim_tienda t ON t.sk_tienda = h.sk_tienda
JOIN dim_producto p ON p.sk_producto = h.sk_producto""").load())

    # --- 2. Ordenar por tienda y categoría (SortRows)
    # Ordena por tienda, categoria
    # NOTA: Spark no garantiza un orden estable entre filas con la misma clave.
    df_ordenar_por_tienda_y_categoria = df_leer_modelo_estrella.orderBy(F.col("tienda").asc_nulls_first(), F.col("categoria").asc_nulls_first())

    # --- 3. Ordenar por tienda (SortRows)
    # Ordena por tienda
    # NOTA: Spark no garantiza un orden estable entre filas con la misma clave.
    df_ordenar_por_tienda = df_leer_modelo_estrella.orderBy(F.col("tienda").asc_nulls_first())

    # --- 4. Ventas por tienda y categoría (GroupBy)
    # Agrupa por tienda, categoria (2 totales)
    # Group by de PDI agrupa filas SEGUIDAS con la misma clave (espera la entrada ordenada).
    d = pdi_rachas(df_ordenar_por_tienda_y_categoria, ["tienda", "categoria"])
    df_ventas_por_tienda_y_categoria = d.groupBy("_racha", "tienda", "categoria").agg(
        F.sum(F.col("cantidad")).alias("unidades"),
        F.sum(F.col("importe")).alias("importe_sin_redondear"),
    ).orderBy("_racha").drop("_racha")

    # --- 5. Totales por tienda (GroupBy)
    # Agrupa por tienda (2 totales)
    # Group by de PDI agrupa filas SEGUIDAS con la misma clave (espera la entrada ordenada).
    d_2 = pdi_rachas(df_ordenar_por_tienda, ["tienda"])
    df_totales_por_tienda = d_2.groupBy("_racha", "tienda").agg(
        F.sum(F.col("importe")).alias("importe_sin_redondear"),
        F.count_distinct(F.col("id_ticket")).alias("num_tickets"),
    ).orderBy("_racha").drop("_racha")

    # --- 6. Redondear importe (Calculator)
    # Calcula dos_decimales, importe
    df_redondear_importe = (df_ventas_por_tienda_y_categoria
        .withColumn("dos_decimales", F.lit(2))
        .withColumn("importe", (F.signum(F.col("importe_sin_redondear")) * F.floor(F.abs(F.col("importe_sin_redondear")).cast("decimal(38,12)") * F.pow(F.lit(10), F.col("dos_decimales")).cast("decimal(38,12)") + 0.5) / F.pow(F.lit(10), F.col("dos_decimales")).cast("decimal(38,12)")).cast("double")))
    df_redondear_importe = df_redondear_importe.drop("dos_decimales")  # campos temporales

    # --- 7. Calcular ticket medio (Calculator)
    # Calcula dos_decimales, importe_total, ticket_sin_redondear, ticket_medio
    df_calcular_ticket_medio = (df_totales_por_tienda
        .withColumn("dos_decimales", F.lit(2))
        .withColumn("importe_total", (F.signum(F.col("importe_sin_redondear")) * F.floor(F.abs(F.col("importe_sin_redondear")).cast("decimal(38,12)") * F.pow(F.lit(10), F.col("dos_decimales")).cast("decimal(38,12)") + 0.5) / F.pow(F.lit(10), F.col("dos_decimales")).cast("decimal(38,12)")).cast("double"))
        .withColumn("ticket_sin_redondear", (F.col("importe_sin_redondear") / F.col("num_tickets")))
        .withColumn("ticket_medio", (F.signum(F.col("ticket_sin_redondear")) * F.floor(F.abs(F.col("ticket_sin_redondear")).cast("decimal(38,12)") * F.pow(F.lit(10), F.col("dos_decimales")).cast("decimal(38,12)") + 0.5) / F.pow(F.lit(10), F.col("dos_decimales")).cast("decimal(38,12)")).cast("double")))
    df_calcular_ticket_medio = df_calcular_ticket_medio.drop("dos_decimales", "ticket_sin_redondear")  # campos temporales

    # --- 8. Cargar KPI tienda y categoría (TableOutput)
    # Escribe en la tabla kpi_ventas_tienda_categoria (dw_grano_de_oro)
    # NOTA: Spark necesita el .jar del driver JDBC (spark.jars).
    df_cargar_kpi_tienda_y_categoria = df_redondear_importe
    (df_redondear_importe.select("tienda", "categoria", "unidades", "importe").write.format("jdbc").options(**jdbc_dw_grano_de_oro)
        .option("dbtable", "kpi_ventas_tienda_categoria")
        .option("batchsize", 1000).option("truncate", True).mode("overwrite").save())

    # --- 9. CSV KPI tienda y categoría (TextFileOutput)
    # Escribe kpi_ventas_tienda_categoria.csv
    # NOTA: Spark escribe una carpeta con varios archivos; aquí se escribe un solo archivo como en PDI, pasando por pandas (junta las filas en el driver).
    df_csv_kpi_tienda_y_categoria = df_redondear_importe
    pdf_csv_kpi_tienda_y_categoria = df_redondear_importe.toPandas()  # junta las filas en el driver
    sal_csv_kpi_tienda_y_categoria = pdf_csv_kpi_tienda_y_categoria[["tienda", "categoria", "unidades", "importe"]].copy()
    ruta_csv_kpi_tienda_y_categoria = _nombre_salida(f"{VARS['Internal.Entry.Current.Directory']}/../salida/kpi_ventas_tienda_categoria", "csv")
    _escribir_texto(sal_csv_kpi_tienda_y_categoria, ruta_csv_kpi_tienda_y_categoria, sep=";", encierro="\"", fin="\n",
                    codificacion="UTF-8", textos=["tienda", "categoria", "unidades", "importe"])
    # en Spark distribuido (una carpeta con varios part-*.csv): df_csv_kpi_tienda_y_categoria.write.mode("overwrite").option("header", True).option("sep", ";").csv(...)

    # --- 10. Cargar KPI ticket medio (TableOutput)
    # Escribe en la tabla kpi_ticket_medio_tienda (dw_grano_de_oro)
    # NOTA: Spark necesita el .jar del driver JDBC (spark.jars).
    df_cargar_kpi_ticket_medio = df_calcular_ticket_medio
    (df_calcular_ticket_medio.select("tienda", "importe_total", "num_tickets", "ticket_medio").write.format("jdbc").options(**jdbc_dw_grano_de_oro)
        .option("dbtable", "kpi_ticket_medio_tienda")
        .option("batchsize", 1000).option("truncate", True).mode("overwrite").save())

    # --- 11. CSV KPI ticket medio (TextFileOutput)
    # Escribe kpi_ticket_medio_tienda.csv
    # NOTA: Spark escribe una carpeta con varios archivos; aquí se escribe un solo archivo como en PDI, pasando por pandas (junta las filas en el driver).
    df_csv_kpi_ticket_medio = df_calcular_ticket_medio
    pdf_csv_kpi_ticket_medio = df_calcular_ticket_medio.toPandas()  # junta las filas en el driver
    sal_csv_kpi_ticket_medio = pdf_csv_kpi_ticket_medio[["tienda", "importe_total", "num_tickets", "ticket_medio"]].copy()
    ruta_csv_kpi_ticket_medio = _nombre_salida(f"{VARS['Internal.Entry.Current.Directory']}/../salida/kpi_ticket_medio_tienda", "csv")
    _escribir_texto(sal_csv_kpi_ticket_medio, ruta_csv_kpi_ticket_medio, sep=";", encierro="\"", fin="\n",
                    codificacion="UTF-8", textos=["tienda", "importe_total", "num_tickets", "ticket_medio"])
    # en Spark distribuido (una carpeta con varios part-*.csv): df_csv_kpi_ticket_medio.write.mode("overwrite").option("header", True).option("sep", ";").csv(...)

## Carga correcta

`SUCCESS`

Success: termina este camino del job con éxito.

In [ ]:
@entrada
def e_carga_correcta():
    """Carga correcta (SUCCESS): Success: termina este camino del job con éxito."""
    return True

## Registrar el error

`WRITE_TO_LOG`

Write to log: escribe un mensaje en el log. Mensaje claro en el log antes de abortar.

In [ ]:
@entrada
def e_registrar_el_error():
    """Registrar el error (WRITE_TO_LOG): Write to log: escribe un mensaje en el log. Mensaje claro en el log antes de abortar."""
    log.error("Grano de Oro - La carga del almacén de Grano de Oro ha fallado: la entrada que ha fallado es la última que aparece como «Starting entry» antes de este mensaje.")
    return True

## Abortar la carga

`ABORT`

Abort job: corta el job entero con error (ningún otro camino sigue). El job termina con error.

In [ ]:
@entrada
def e_abortar_la_carga():
    """Abortar la carga (ABORT): Abort job: corta el job entero con error (ningún otro camino sigue). El job termina con error."""
    raise Abortar("Carga abortada: el almacén puede haber quedado a medias. Corrige el error y vuelve a lanzar el job.")

## Registrar el error

`rama`

Sigue el flujo desde «Registrar el error» y devuelve su resultado.

In [ ]:
def rama_registrar_el_error():
    """Registrar el error: Sigue el flujo desde «Registrar el error» y devuelve su resultado."""
    e_registrar_el_error()
    return e_abortar_la_carga()

## Flujo del job

main() sigue los hops desde START: los verdes si la entrada va bien, los rojos si falla, los negros siempre.

In [ ]:
def main():
    """Flujo del job «job_carga_dw»: True si termina bien (como el código 0 de Kitchen)."""
    try:
        if not e_comprobar_ficheros_de_entrada():
            return rama_registrar_el_error()
        if not e_crear_carpeta_salida():
            return rama_registrar_el_error()
        if not e_crear_modelo_estrella():
            return rama_registrar_el_error()
        if not tr_cargar_dim_tienda():
            return rama_registrar_el_error()
        if not tr_cargar_dim_producto():
            return rama_registrar_el_error()
        if not tr_cargar_dim_fecha():
            return rama_registrar_el_error()
        if not tr_cargar_hechos_ventas():
            return rama_registrar_el_error()
        if not tr_calcular_kpi():
            return rama_registrar_el_error()
        return e_carga_correcta()
    except Abortar as e:
        log.error(str(e))
        return False

## Ejecutar el job

In [ ]:
ok = main()
print("Resultado del job: ", "OK" if ok else "ERROR")